In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import joblib

In [2]:
df = pd.read_csv(r'C:\Users\provu\Desktop\my_ml_service\StudentPerformanceFactors.csv')

# Convert Exam_Score (numeric) into a binary classification target
# High = score >= 70, Low = score < 70
df['performance'] = df['Exam_Score'].apply(lambda x: 'High' if x >= 70 else 'Low')

# Separate features and target
x_cols = [c for c in df.columns if c not in ['Exam_Score', 'performance']]
X = df[x_cols]
y = df['performance']

print(f"Shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
df.head()

Shape: (6607, 19)
Target distribution:
performance
Low     4982
High    1625
Name: count, dtype: int64


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,...,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score,performance
0,23,84,Low,High,No,7,73,Low,Yes,0,...,Medium,Public,Positive,3,No,High School,Near,Male,67,Low
1,19,64,Low,Medium,No,8,59,Low,Yes,2,...,Medium,Public,Negative,4,No,College,Moderate,Female,61,Low
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,...,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74,High
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,...,Medium,Public,Negative,4,No,High School,Moderate,Male,71,High
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,...,High,Public,Neutral,4,No,College,Near,Female,70,High


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1234
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (4624, 19), Test: (1983, 19)


In [4]:
train_mode = dict(X_train.mode().iloc[0])
X_train = X_train.fillna(train_mode)
print("Fill values:", train_mode)

Fill values: {'Hours_Studied': np.int64(19), 'Attendance': np.int64(76), 'Parental_Involvement': 'Medium', 'Access_to_Resources': 'Medium', 'Extracurricular_Activities': 'Yes', 'Sleep_Hours': np.int64(7), 'Previous_Scores': np.int64(66), 'Motivation_Level': 'Medium', 'Internet_Access': 'Yes', 'Tutoring_Sessions': np.int64(1), 'Family_Income': 'Medium', 'Teacher_Quality': 'Medium', 'School_Type': 'Public', 'Peer_Influence': 'Positive', 'Physical_Activity': np.int64(3), 'Learning_Disabilities': 'No', 'Parental_Education_Level': 'High School', 'Distance_from_Home': 'Near', 'Gender': 'Male'}


In [5]:
encoders = {}
categorical_columns = [
    'Parental_Involvement', 'Access_to_Resources',
    'Extracurricular_Activities', 'Motivation_Level',
    'Internet_Access', 'Family_Income', 'Teacher_Quality',
    'School_Type', 'Peer_Influence', 'Learning_Disabilities',
    'Parental_Education_Level', 'Distance_from_Home', 'Gender'
]
for column in categorical_columns:
    le = LabelEncoder()
    X_train[column] = le.fit_transform(X_train[column])
    encoders[column] = le

print("Encoders created for:", list(encoders.keys()))

Encoders created for: ['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf = rf.fit(X_train, y_train)
print("Random Forest trained!")

# Quick accuracy check on test set
X_test_clean = X_test.fillna(train_mode)
for col in categorical_columns:
    X_test_clean[col] = encoders[col].transform(X_test_clean[col])
print(f"Test accuracy: {rf.score(X_test_clean, y_test):.4f}")
```

### Cell 7 — Train Extra Trees

```python
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et = et.fit(X_train, y_train)
print("Extra Trees trained!")
print(f"Test accuracy: {et.score(X_test_clean, y_test):.4f}")

SyntaxError: invalid syntax (1755167363.py, line 10)

In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf = rf.fit(X_train, y_train)
print("Random Forest trained!")

# Quick accuracy check on test set
X_test_clean = X_test.fillna(train_mode)
for col in categorical_columns:
    X_test_clean[col] = encoders[col].transform(X_test_clean[col])
print(f"Test accuracy: {rf.score(X_test_clean, y_test):.4f}")
```

### Cell 7 — Train Extra Trees

```python
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et = et.fit(X_train, y_train)
print("Extra Trees trained!")
print(f"Test accuracy: {et.score(X_test_clean, y_test):.4f}")

SyntaxError: invalid syntax (1755167363.py, line 10)

In [8]:
joblib.dump(train_mode, "./train_mode.joblib",   compress=True)
joblib.dump(encoders,   "./encoders.joblib",      compress=True)
joblib.dump(rf,         "./random_forest.joblib", compress=True)
joblib.dump(et,         "./extra_trees.joblib",   compress=True)
print("All 4 files saved in research/!")
```

After running Cell 8, your `research/` folder contains:
```
research/
├── train_models.ipynb
├── train_mode.joblib       ← fill-missing values
├── encoders.joblib         ← label encoders for 13 categorical columns
├── random_forest.joblib    ← trained RF model
└── extra_trees.joblib      ← trained ET model

SyntaxError: invalid character '├' (U+251C) (415080526.py, line 11)